## Deep Research Agent — LangGraph

Agentic research pipeline built with **LangGraph** and open-source models (Groq / Cerebras / OpenRouter).

**File structure:**
- `models.py` — Pydantic schemas + `ResearchState` TypedDict
- `tools.py` — `web_search` (DuckDuckGo) + `send_report_email` (SendGrid)
- `agent.py` — LLM setup, guardrails, all node functions, routing, graph construction, helper coroutines
- `app.py` — Gradio UI (also runnable as `python app.py`)

**Pipeline:** Clarify → Plan → Search (parallel) → Sufficiency check → Write → Evaluate → Email

**Key features:**
- SQLite persistent memory — sessions survive kernel restarts
- Per-user session isolation via unique `thread_id` in `gr.State`
- Session resume — paste a previous session ID to load a prior report
- Writer output streamed live token-by-token via `astream_events`
- LLM-based input guardrail + heuristic output guardrail
- Explicit model parameters per agent role (`temperature`, `max_tokens`, `top_p`)
- LangSmith tracing via `LANGCHAIN_API_KEY` (no inference tokens used)
- Provider switchable via `RESEARCH_PROVIDER` env var: `groq` | `cerebras` | `openrouter` | `openai`

In [ ]:
!uv pip install langgraph langchain-openai langchain-community langchain-groq ddgs sendgrid gradio python-dotenv pydantic nest-asyncio

In [ ]:
import sys, os

# Ensure local modules (models.py, tools.py, agent.py, app.py) are importable
# when the notebook is opened from a different working directory
_nb_dir = os.path.dirname(os.path.abspath("deep_research_langgraph.ipynb"))
if _nb_dir not in sys.path:
    sys.path.insert(0, _nb_dir)

# Set your provider here or via RESEARCH_PROVIDER in .env
# Options: groq | cerebras | openrouter | openai
os.environ.setdefault("RESEARCH_PROVIDER", "groq")

# Optional: LangSmith tracing (free, uses LANGCHAIN_API_KEY, no inference tokens)
# os.environ["LANGCHAIN_API_KEY"] = "ls__..."

print(f"Provider: {os.environ['RESEARCH_PROVIDER']}")

In [ ]:
from IPython.display import Image, display
from agent import research_graph, PROVIDER, _cfg

print(f"Provider: {PROVIDER} | Model: {_cfg['model']}")
display(Image(research_graph.get_graph().draw_mermaid_png()))

In [ ]:
from app import demo
demo.launch(inbrowser=True)